In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Load dataset
df = pd.read_csv("engine training data.csv")

# Define Features and Target
features = ['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant pressure', 'Lub oil temp', 'Coolant temp']
target = 'Engine Condition'  # 0 = Anomaly, 1 = Normal

# Drop missing values
df = df.dropna(subset=features + [target])

# Standardize data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])
joblib.dump(scaler, "scaler.pkl")  # Save scaler for future use

# Define target variable
y = df[target].values

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=500, random_state=42, class_weight="balanced")
rf_model.fit(X_train, y_train)

# Save the trained model
joblib.dump(rf_model, "engine_rf_model.pkl")

# Model Evaluation
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")
print("Classification Report:\n", report)

# Predict Anomalies in the entire dataset
df["Anomaly"] = rf_model.predict(X_scaled) == 0  # Mark anomalies as 'True' if predicted as class 0
print(f"Anomalies detected: {df['Anomaly'].sum()}")

# Compute threshold values dynamically for normal engine conditions
normal_data = df[df[target] == 1]
thresholds = {col: (normal_data[col].min(), normal_data[col].max()) for col in features}

# Save thresholds for real-time detection
joblib.dump(thresholds, "thresholds.pkl")



Model Accuracy: 0.66
Classification Report:
               precision    recall  f1-score   support

           0       0.55      0.37      0.44      1095
           1       0.69      0.82      0.75      1905

    accuracy                           0.66      3000
   macro avg       0.62      0.60      0.60      3000
weighted avg       0.64      0.66      0.64      3000

Anomalies detected: 5173


['thresholds.pkl']

In [ ]:
from collections import Counter
print(Counter(y_pred))


Counter({np.int64(1): 1689, np.int64(0): 1311})
